# 05 · Duplicate validation: EICV7 section 9

Notebook 04 found repeated rows in four EICV7 files. This notebook finds out why, sorts every repeat into a class, applies one documented dedup rule, and saves clean copies for the household master table.

| | |
|---|---|
| **Inputs** | `Microdata/Cross_Section/CS_S9A_transfers_out.dta`, `CS_S9B_transfers_in.dta`, `CS_S9C_other_expenditure.dta`, `CS_S9E_other_income.dta`. Read only, never modified. |
| **Outputs** | `data/interim/s9{a,b,c,e}_*_dedup.parquet` (clean files), `data/interim/s9_dedup_dropped_rows.parquet` (every removed row and why), `data/interim/s9_dedup_household_flags.parquet` (one row per household) |
| **Code** | `src/data/deduplicate_s9.py` |
| **Report** | `reports/duplicate_validation.md` |
| **Runtime** | About 15 seconds |

**Rule in one line.** Remove exact repeats. Remove "no transfers" rows that contradict listed transfers. Where one key carries two different records, keep the first row; if the two records disagree on an amount, mark that household's total for that file as unusable rather than guessing which record is right. No value in a kept row is changed.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data" / "deduplicate_s9.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.data import deduplicate_s9 as dd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
SPECS = dd.SPECS
SHORT = {"transfers_out": "S9A", "transfers_in": "S9B", "other_expenditure": "S9C", "other_income": "S9E"}

## 1. Expected structure

The questionnaire fixes what one row should be, and therefore the key. Page numbers are PDF pages of `docs/source_documentation/eicv7/EICV7_Final_Questionnaire_English.pdf`.

| File | One row is | Key | Questionnaire |
|---|---|---|---|
| S9A transfers out | one recipient listed by the household (`s9aq2a` = recipient line number); a single "no" row (`s9aq0` = 2) if the household sent nothing | `hhid` + `s9aq2a` | PDF p. 60, Part A |
| S9B transfers in | one sender (`s9bq3` = sender line number); a single "no" row (`s9bq0` = 2) if nothing was received | `hhid` + `s9bq3` | PDF p. 61, Part B |
| S9C other expenditure | one of 13 fixed items (taxes, contributions, ceremonies), asked of every household | `hhid` + `s9cq1` | PDF p. 62, Part C, codes 1–13 |
| S9E other income | one of 26 fixed income sources, asked of every household | `hhid` + `s9eq1` | PDF p. 70, codes 01–26 |

Several rows per household are therefore normal in all four files: several recipients or senders in S9A and S9B, and 13 or 26 items in S9C and S9E. Those are **legitimate multiple items**. A duplicate is a second row with the **same key**.

In [ ]:
raw = {name: dd.load_raw(spec) for name, spec in SPECS.items()}
rows = []
for name, d in raw.items():
    spec = SPECS[name]
    per_hh = d.groupby("hhid").size()
    content = [c for c in d.columns if c != "row_in_file"]
    rows.append({
        "file": SHORT[name], "rows": len(d), "households": d["hhid"].nunique(),
        "rows/hh median": per_hh.median(), "rows/hh max": per_hh.max(),
        "rows sharing a key": int(d.duplicated(spec.key, keep=False).sum()),
        "exact duplicate rows (beyond first)": int(d.duplicated(content).sum()),
    })
profile = pd.DataFrame(rows).set_index("file")
profile

Every file covers all 15,054 households. S9C and S9E should have exactly 13 and 26 rows per household, but some households have 26 and 52. S9A and S9B have households with 99 rows.

## 2. Classify the duplicates

`dd.run()` applies the rule in four steps and labels every raw row. The labels are used below to describe the duplicates; section 3 states the rule.

In [ ]:
res = dd.run(write=True)
ann, clean, dropped, flags = res["annotated"], res["clean"], res["dropped"], res["flags"]

def classify(name):
    d = ann[name]
    kept = d[d["drop_reason"].isna()]
    keyed = kept[kept[SPECS[name].item].notna()]
    multi_hh = keyed.groupby("hhid").size()
    return {
        "raw rows": len(d),
        "kept: several items in hh (legitimate)": int(keyed["hhid"].isin(multi_hh[multi_hh > 1].index).sum()),
        "kept: single row in hh": int(len(kept) - keyed["hhid"].isin(multi_hh[multi_hh > 1].index).sum()),
        "dropped: exact repeat": int(d["drop_reason"].eq("exact_repeat").sum()),
        "dropped: contradictory 'no' row": int(d["drop_reason"].eq("contradictory_no_row").sum()),
        "dropped: key conflict (amounts agree)": int((d["drop_reason"].eq("key_conflict") & d["key_conflict"].eq("non_amount")).sum()),
        "dropped: key conflict (amounts differ)": int((d["drop_reason"].eq("key_conflict") & d["key_conflict"].eq("amount")).sum()),
        "clean rows": len(clean[name]),
        "households with exact repeats": int(d.loc[d["drop_reason"].eq("exact_repeat"), "hhid"].nunique()),
        "households not usable for totals": int(d.loc[~d["use_in_totals"], "hhid"].nunique()),
    }

classes = pd.DataFrame({SHORT[n]: classify(n) for n in SPECS})
assert (classes.loc["raw rows"] == classes.loc[[i for i in classes.index if i.startswith(("kept", "dropped"))]].sum()).all()
classes

### 2.1 Exact repeats: the same households in every file

The exact repeats are not spread at random. The same households have their whole section 9 written out twice, in all four files.

In [ ]:
rep_hh = {SHORT[n]: set(ann[n].loc[ann[n]["drop_reason"].eq("exact_repeat"), "hhid"].astype(int)) for n in SPECS}
overlap = pd.DataFrame({a: {b: len(rep_hh[a] & rep_hh[b]) for b in rep_hh} for a in rep_hh})
print("Households with exact repeats, and how many of them also repeat in the other files:")
display(overlap)

copies = pd.DataFrame({SHORT[n]: ann[n].groupby("hhid")["n_raw_copies"].max().value_counts().sort_index() for n in SPECS}).fillna(0).astype(int)
copies.index.name = "max copies of a row in the household"
print("Households by the largest number of identical copies of one row:")
display(copies)

In [ ]:
# The 99-copy households: what is repeated?
for n in ["transfers_out", "transfers_in"]:
    d, flag = ann[n], SPECS[n].flag
    big = d.loc[d["n_raw_copies"] >= 50]
    kept_big = big[big["drop_reason"].isna()]
    print(f"{SHORT[n]}: {big['hhid'].nunique()} households have a row repeated 99 times; "
          f"the repeated row is a '{'no' if kept_big[flag].eq(2).all() else 'mixed'}' row "
          f"with {int(kept_big[list(SPECS[n].amounts)].notna().sum().sum())} amounts filled")

In [ ]:
# Where are the doubled households? Doubled = whole S9C list repeated (26 rows instead of 13).
pov = pd.read_stata(dd.EICV7_DIR / "CS_EICV7_poverty_file.dta", columns=["hhid", "clust", "district", "ur", "weight"])
pov["hhid"] = pov["hhid"].astype(np.int64)
doubled = set(flags.loc[flags["other_expenditure_raw_rows"].eq(26), "hhid"])
pov["doubled"] = pov["hhid"].isin(doubled)
by_clust = pov.groupby("clust", observed=True).agg(households=("hhid", "size"), doubled=("doubled", "sum"))
hit = by_clust[by_clust["doubled"] > 0]
print(f"{len(doubled)} doubled households in {len(hit)} of {len(by_clust)} clusters; "
      f"in those clusters {hit['doubled'].sum()} of {hit['households'].sum()} households are doubled "
      f"(median {hit['doubled'].median():.0f} of {hit['households'].median():.0f} per cluster).")
print(f"Weighted share of households doubled: {pov.loc[pov['doubled'], 'weight'].sum() / pov['weight'].sum():.2%}")
dist = (pov[pov["doubled"]].groupby(["district", "ur"], observed=True).size().rename("doubled households")
        .reset_index().sort_values("doubled households", ascending=False))
dist

### 2.2 Version conflicts: two different records under one `hhid`

After exact repeats are removed, S9C and S9E still contain repeated item codes for four households. In a fixed-list file each item can appear only once. Here it appears twice with different amounts, so each household has **two different versions of section 9**. The same households also have conflicting recipient and sender rows in S9A and S9B.

In [ ]:
vc = sorted(res["version_conflict_hh"])
def version_totals(name):
    d, spec = ann[name], SPECS[name]
    x = d[d["hhid"].astype(int).isin(vc) & d["drop_reason"].fillna("").ne("exact_repeat")].copy()
    x["version"] = x.groupby(["hhid", spec.item]).cumcount() + 1
    amt = spec.amounts[-1]
    return x.pivot_table(index=x["hhid"].astype(int), columns="version", values=amt, aggfunc="sum").add_prefix(f"{SHORT[name]} {amt} v")

vt = pd.concat([version_totals("other_expenditure"), version_totals("other_income")], axis=1)
print("Totals (RWF, 12 months) by version, for items where the versions differ or agree:")
display(vt)
print("Version 1 is the first occurrence in file order. The file gives no way to tell which version belongs to the household in the poverty file.")

In [ ]:
# Row-level key conflicts outside the four version-conflict households (S9A only).
other = []
for n in SPECS:
    d, spec = ann[n], SPECS[n]
    g = d[d["key_conflict"].ne("none") & ~d["hhid"].astype(int).isin(vc)]
    for (h, item), grp in g.groupby(["hhid", spec.item], dropna=False):
        content = [c for c in raw[n].columns if c not in ("row_in_file",)]
        diff = [c for c in content if grp[c].nunique(dropna=False) > 1]
        other.append({"file": SHORT[n], "hhid": int(h), "item": item, "rows": len(grp),
                      "type": grp["key_conflict"].iloc[0], "differing variables": ", ".join(diff),
                      "use_in_totals": bool(grp["use_in_totals"].all())})
pd.DataFrame(other)

### 2.3 Contradictory "no" rows

In S9A and S9B, a household that sent or received nothing has one row with the filter question answered "no". A few households have a "no" row **and** a list of recipients or senders. These "no" rows carry no amounts, so dropping them changes no total.

In [ ]:
nr = dropped[dropped["drop_reason"].eq("contradictory_no_row")]
nr.groupby("file").agg(rows=("hhid", "size"), households=("hhid", "nunique"), hhids=("hhid", lambda s: sorted(s.unique())))

## 3. The dedup rule

Applied by `src/data/deduplicate_s9.py`, in this order:

| Step | Rows affected | Action | Effect on totals |
|---|---|---|---|
| 1. **Exact repeat** | a row identical to an earlier row on every variable | drop; keep the first copy | removes double counting |
| 2. **Contradictory "no" row** (S9A, S9B) | "no transfers" row in a household that also lists transfers | drop | none (no amounts) |
| 3a. **Key conflict, amounts agree** | same key as an earlier row, different non-amount content | drop the later row; keep the first | none |
| 3b. **Key conflict, amounts differ** | same key, different amounts | drop the later row; keep the first; set `use_in_totals = False` for the household in that file | household total treated as missing |
| 4. **Version conflict** | household with a 3b conflict in S9C or S9E (fixed lists) | set `use_in_totals = False` for the household in **all four** files | household totals treated as missing |

Why step 4 covers all four files: two versions of the whole section exist, and in the roster files (S9A, S9B) the rows of the two versions cannot be told apart. Treating the household as missing is preferred to picking a version at random.

**Using the clean files.** Every clean file is unique on its key. Before summing amounts to household level, keep rows with `use_in_totals == True`. For households where it is False, set the household total to missing, not zero. `data/interim/s9_dedup_household_flags.parquet` carries the same flags at household level.

## 4. Checks on the clean files

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "result": "pass" if ok else "FAIL", "detail": detail})

all_hh = set(pov["hhid"])
for n, c in clean.items():
    spec, s = SPECS[n], SHORT[n]
    check(f"{s}: unique on key", not c.duplicated(spec.key).any())
    check(f"{s}: covers all 15,054 households", set(c["hhid"].astype(int)) == all_hh)
    check(f"{s}: kept + dropped = raw rows", len(c) + (dropped["file"] == spec.file).sum() == len(raw[n]))
    merged = c[raw[n].columns].merge(raw[n], on="row_in_file", suffixes=("", "_raw"))
    same = all(merged[col].equals(merged[f"{col}_raw"]) for col in raw[n].columns if col != "row_in_file")
    check(f"{s}: kept rows identical to raw rows", same and len(merged) == len(c))
    if spec.fixed_list:
        sizes = c.groupby("hhid").size()
        n_items = c[spec.item].nunique()
        check(f"{s}: exactly {n_items} items per household", sizes.eq(n_items).all(), f"{n_items} item codes")
    else:
        f = spec.flag
        mixed = c.groupby("hhid")[f].nunique().gt(1).sum()
        no_rows = c[c[f].eq(2)].groupby("hhid").size().max()
        check(f"{s}: no household mixes 'no' and listed transfers", mixed == 0)
        check(f"{s}: at most one 'no' row per household", no_rows == 1)
check("household flags: one row per household", flags["hhid"].is_unique and len(flags) == 15_054)
check("version-conflict households flagged in all four files",
      not flags.loc[flags["s9_version_conflict"], [f"{n}_use_in_totals" for n in SPECS]].any().any())
checks = pd.DataFrame(checks)
assert checks["result"].eq("pass").all(), checks[checks["result"] != "pass"]
checks

## 5. Totals before and after deduplication

Weighted totals use the household weight `weight` and are national annual totals for the 12-month amounts (S9A, S9B, S9C, S9E `s9eq4`). `s9eq3` is the 4-week amount and is shown only for comparison. Stage 1 removes exact repeats only; stage 2 is the clean file; stage 3 keeps only households usable for totals.

In [ ]:
st = pd.concat([dd.stage_totals(ann[n], SPECS[n]) for n in SPECS], ignore_index=True)
st["file"] = st["file"].map(SHORT)
st["weighted, RWF bn"] = st["weighted_sum_rwf"] / 1e9
base = st[st["stage"].eq("0 raw")].set_index(["file", "amount"])["weighted_sum_rwf"]
st["change vs raw, %"] = st.apply(lambda r: 100 * (r["weighted_sum_rwf"] / base[(r["file"], r["amount"])] - 1), axis=1)
view = st.pivot_table(index=["file", "amount"], columns="stage", values="weighted, RWF bn", sort=False)
view["% change, raw to stage 2"] = st[st["stage"].str.startswith("2")].set_index(["file", "amount"])["change vs raw, %"]
view["% change, raw to stage 3"] = st[st["stage"].str.startswith("3")].set_index(["file", "amount"])["change vs raw, %"]
view

In [ ]:
# Household-level effect: for the households that had exact repeats, how much did their totals shrink?
def hh_total(d, name, mask):
    x = d.loc[mask, dd.ANNUAL_AMOUNTS[name]].sum(axis=1)
    return x.groupby(d.loc[mask, "hhid"].astype(int)).sum()

rows = []
for n in SPECS:
    d, spec = ann[n], SPECS[n]
    h = sorted(set(d.loc[d["drop_reason"].eq("exact_repeat"), "hhid"].astype(int)) - set(vc))
    before = hh_total(d, n, pd.Series(True, index=d.index)).reindex(h)
    after = hh_total(d, n, d["drop_reason"].isna()).reindex(h)
    pos = before > 0
    ratio = (before[pos] / after[pos])
    rows.append({"file": SHORT[n], "12-month amounts": " + ".join(dd.ANNUAL_AMOUNTS[n]), "households with repeats": len(h),
                 "of which with a positive total": int(pos.sum()),
                 "raw / clean total: median": ratio.median(), "min": ratio.min(), "max": ratio.max()})
pd.DataFrame(rows).set_index("file")

For the households with repeats, the raw total is exactly **twice** the clean total: summing the raw rows double-counts each of them. The national totals move by only 0.6–2.3% (4% for the 4-week income amount), because these households are about 1.3% of all households (weighted). The error would be large in any household-level analysis, though, which is what the master table is for.

In [ ]:
# By item, for the fixed-list files.
item_labels = {n: dd.item_labels(SPECS[n]) for n in ["other_expenditure", "other_income"]}

tabs = []
for n in ["other_expenditure", "other_income"]:
    d, spec = ann[n], SPECS[n]
    amt = spec.amounts[-1]
    w = lambda m: (d.loc[m, amt] * d.loc[m, "weight"]).groupby(d.loc[m, spec.item]).sum() / 1e9
    t = pd.DataFrame({"raw, RWF bn": w(pd.Series(True, index=d.index)),
                      "clean (stage 3), RWF bn": w(d["drop_reason"].isna() & d["use_in_totals"])})
    t["% change"] = 100 * (t.iloc[:, 1] / t.iloc[:, 0] - 1)
    t.index = [f"{SHORT[n]} {int(i):02d} {item_labels[n].get(int(i), '')}"[:60] for i in t.index]
    tabs.append(t)
items = pd.concat(tabs)
items

In [ ]:
# Prevalence (weighted share of households with a positive 12-month amount) before and after.
rows = []
for n in SPECS:
    d, spec = ann[n], SPECS[n]
    vals = d[dd.ANNUAL_AMOUNTS[n]].sum(axis=1)
    hh_raw = (vals > 0).groupby(d["hhid"].astype(int)).any()
    keep = d["drop_reason"].isna() & d["use_in_totals"]
    hh_clean = (vals[keep] > 0).groupby(d.loc[keep, "hhid"].astype(int)).any()
    wt = pov.set_index("hhid")["weight"]
    rows.append({"file": SHORT[n], "raw, %": 100 * np.average(hh_raw, weights=wt.reindex(hh_raw.index)),
                 "clean, %": 100 * np.average(hh_clean, weights=wt.reindex(hh_clean.index)),
                 "households in denominator (clean)": len(hh_clean)})
prev = pd.DataFrame(rows).set_index("file")
prev

Prevalence barely moves, because a repeat never turns a zero into a positive amount. Deduplication changes **how much**, not **whether**.

## 6. Saved outputs

In [ ]:
for p in sorted(dd.INTERIM_DIR.glob("s9*.parquet")):
    x = pd.read_parquet(p)
    print(f"{p.relative_to(ROOT)}: {len(x):,} rows x {x.shape[1]} columns, {p.stat().st_size / 1e6:.1f} MB")
print()
print("Dropped-row log by reason:")
display(dropped.groupby(["file", "drop_reason"]).size().unstack(fill_value=0))
print("Household flags: columns", list(flags.columns))

## 7. Findings

1. **All four files share one problem.** The households whose whole section 9 was written out twice are the same households in every file: 222 in S9C and S9E. They are clustered: 32 enumeration areas, with 7 of 9 sampled households in the median one, and 108 of the 222 in rural Rutsiro. The cause is not documented; it looks like a data-processing repeat, not a survey design feature.
2. **The largest repeats are harmless placeholders.** 23 households have their single "no transfers" row repeated 99 times in S9B; one of them ([ID withheld]) also has it in S9A. These rows hold no amounts, so they inflate row counts, not totals.
3. **Doubling matters at household level.** For every doubled household with a positive amount, the raw total was exactly twice the clean total. National totals fall by only 0.6–2.3%.
4. **Four households have two different versions of section 9** (`hhid` [ID withheld], [ID withheld], [ID withheld], [ID withheld]). It is impossible to tell which version is right, so their section 9 totals are marked unusable (`use_in_totals = False`) and must be treated as missing. One more household ([ID withheld]) has a single conflicting transfer-out row and is treated the same way in S9A only.
5. **No legitimate multiple items were removed.** Several recipients, senders or items per household are kept. Only rows repeating an existing key were dropped, and every removed row is logged in `data/interim/s9_dedup_dropped_rows.parquet`.